In [1]:
# Top cell: install the libraries this notebook uses.
# If they are already installed, pip will simply say so.
%pip install -q pandas matplotlib numpy scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
ROOT = Path('..').resolve()
PROC = ROOT / 'data' / 'processed'
CH = ROOT / 'charts'
CH.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 150})
print('Paths ready:', PROC.exists(), CH.exists())

Paths ready: True True


# 4. Regional patterns: not everywhere warms at the same pace

Plain words first. A hemisphere is one half of the Earth: northern or southern. A latitude band is a strip around the globe, for example 64N to 90N, which is the far north including the Arctic. Warming is not uniform. Some places warm faster than the global average. This notebook compares bands, it does not draw a map, because no gridded map dataset was used in this build.

Source: NASA GISTEMP zonal means, 1880 to 2025. For comparisons with the repo global file, years to 2023 are used. The 2024 and 2025 zonal values are reported separately in the text.

In [3]:
z = pd.read_csv(PROC / 'zonal_annual.csv')
print(z.head().to_string(index=False))
print('Years:', z['year'].min(), 'to', z['year'].max())
print('Global 2024:', z[z['year']==2024]['global_c'].iloc[0], '| 2025:', z[z['year']==2025]['global_c'].iloc[0])

 year  global_c  northern_hemisphere_c  southern_hemisphere_c  north_24_to_90_c  tropics_24s_to_24n_c  south_90_to_24_c  arctic_64_to_90_c
 1880     -0.17                  -0.30                  -0.05             -0.39                 -0.12             -0.02              -0.76
 1881     -0.09                  -0.19                   0.00             -0.36                  0.11             -0.07              -0.88
 1882     -0.11                  -0.22                  -0.01             -0.32                 -0.04              0.01              -1.37
 1883     -0.18                  -0.29                  -0.06             -0.36                 -0.16             -0.01              -0.15
 1884     -0.28                  -0.43                  -0.14             -0.62                 -0.14             -0.14              -1.28
Years: 1880 to 2025
Global 2024: 1.28 | 2025: 1.18


In [4]:
z23 = z[z['year'] <= 2023]
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z23['year'], z23['northern_hemisphere_c'], label='Northern hemisphere', color='#d62728')
ax.plot(z23['year'], z23['southern_hemisphere_c'], label='Southern hemisphere', color='#1f77b4')
ax.plot(z23['year'], z23['global_c'], label='Global', color='black', linewidth=1)
ax.axhline(0, color='grey', linewidth=0.8)
ax.legend()
ax.set_title('Northern hemisphere warmed more than southern hemisphere, 1880 to 2023')
ax.set_xlabel('Year')
ax.set_ylabel('Anomaly in degrees Celsius')
fig.savefig(CH / '13_hemisphere_comparison.png', bbox_inches='tight')
plt.show()

In [5]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z23['year'], z23['arctic_64_to_90_c'], label='64N to 90N (far north)', color='#d62728')
ax.plot(z23['year'], z23['global_c'], label='Global', color='black', linewidth=1)
ax.axhline(0, color='grey', linewidth=0.8)
ax.legend()
ax.set_title('The far north warmed much faster than the globe as a whole')
ax.set_xlabel('Year')
ax.set_ylabel('Anomaly in degrees Celsius')
fig.savefig(CH / '14_arctic_vs_global.png', bbox_inches='tight')
plt.show()

In [6]:
recent = z23[z23['year'].between(2014, 2023)]
early = z23[z23['year'].between(1880, 1889)]
for col in ['global_c', 'northern_hemisphere_c', 'southern_hemisphere_c', 'arctic_64_to_90_c', 'tropics_24s_to_24n_c']:
    print(col, '| 1880-1889 mean:', round(early[col].mean(), 2), '| 2014-2023 mean:', round(recent[col].mean(), 2), '| rise:', round(recent[col].mean() - early[col].mean(), 2))
print('Arctic rise divided by global rise:', round((recent['arctic_64_to_90_c'].mean() - early['arctic_64_to_90_c'].mean()) / (recent['global_c'].mean() - early['global_c'].mean()), 2))

global_c | 1880-1889 mean: -0.21 | 2014-2023 mean: 0.93 | rise: 1.14
northern_hemisphere_c | 1880-1889 mean: -0.32 | 2014-2023 mean: 1.19 | rise: 1.51
southern_hemisphere_c | 1880-1889 mean: -0.11 | 2014-2023 mean: 0.67 | rise: 0.78
arctic_64_to_90_c | 1880-1889 mean: -1.11 | 2014-2023 mean: 2.44 | rise: 3.56
tropics_24s_to_24n_c | 1880-1889 mean: -0.08 | 2014-2023 mean: 0.81 | rise: 0.89
Arctic rise divided by global rise: 3.11


In [7]:
g = pd.read_csv(PROC / 'global_annual.csv')
early_s = g[g['year'].between(1880, 1889)][['winter_djf_c', 'spring_mam_c', 'summer_jja_c', 'autumn_son_c']].mean()
recent_s = g[g['year'].between(2014, 2023)][['winter_djf_c', 'spring_mam_c', 'summer_jja_c', 'autumn_son_c']].mean()
print('Season means, 1880-1889:'); print(early_s.round(2).to_string())
print('Season means, 2014-2023:'); print(recent_s.round(2).to_string())
labels = ['Winter (DJF)', 'Spring (MAM)', 'Summer (JJA)', 'Autumn (SON)']
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(labels))
ax.bar([i - 0.2 for i in x], early_s.values, width=0.4, label='1880-1889 mean', color='#bbbbbb')
ax.bar([i + 0.2 for i in x], recent_s.values, width=0.4, label='2014-2023 mean', color='#d62728')
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_xticks(list(x)); ax.set_xticklabels(labels); ax.legend()
ax.set_title('Every season is warmer in 2014-2023 than in 1880-1889')
ax.set_ylabel('Anomaly in degrees Celsius')
fig.savefig(CH / '15_seasonal_comparison.png', bbox_inches='tight')
plt.show()

Season means, 1880-1889:
winter_djf_c   -0.24
spring_mam_c   -0.20
summer_jja_c   -0.20
autumn_son_c   -0.21
Season means, 2014-2023:
winter_djf_c    0.92
spring_mam_c    0.95
summer_jja_c    0.87
autumn_son_c    0.96


## What this shows

From the 1880s to 2014-2023, the global mean rose by about 1.14 degrees, the northern hemisphere by about 1.51 degrees, the southern hemisphere by about 0.78 degrees, and the far north band (64N to 90N) by about 3.55 degrees. That far north rise is about 3.1 times the global rise. Scientists call faster Arctic warming Arctic amplification. This project reports the ratio from the data; it does not add a separate physical explanation beyond that label. Every season rose too: winter, spring, summer, and autumn are all warmer in 2014-2023 than in 1880-1889.